# 01 — CardSim calibration probe

Derivation notebook for `scripts/probe_cardsim.py`. It documents the pilot
simulation that precedes the full-scale dataset build, and its purpose is
narrow on three counts:

1. **Verify the installation.** The simulator must run against the local DCPC
   CSVs bundled with the `cardsim` package. The Atlanta Fed remote endpoint
   changed format and breaks the default download path, so offline operation is
   a requirement, not a convenience.
2. **Measure what the simulator does not expose.** CardSim has no input
   parameter for transaction frequency per card-day. That quantity emerges from
   the DCPC calibration and must be estimated empirically before the base
   dataset can be sized (Eq. 3 and Eq. 4 of the technical notes).
3. **Inspect the raw schema.** CardSim returns 13 columns in simulator
   vocabulary. Enumerating them here fixes the exact inventory that
   `normalize_cardsim_df()` maps to the 11-column canonical schema.

The simulation itself lives in `fraud_stat_engine.probe.run_probe()`. This
notebook imports it rather than re-implementing it, so the assertion on `f̂`
below guards the behaviour of the script that actually produces data — not a
notebook-local copy that could silently drift from it.

**Out of scope.** This notebook does not generate the project dataset, and says
nothing about merchant-level traffic, day-of-week effects, or the 209,185 rows
of `cardsim_base.parquet`. That is `02_eda_cardsim_base.ipynb`.

In [4]:
import math
from pathlib import Path

import cardsim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cardsim import Cardsim

plt.rcParams["figure.dpi"] = 110

# Use bundled DCPC calibration CSVs; the Atlanta Fed remote endpoint changed
# format and breaks CardSim's default download behavior.
DCPC_DIR = str(Path(cardsim.__file__).resolve().parent / "dcpc")

SEED = 42
FRAUD_RATE = 0.01
FRAUD_FLAG_THRESHOLD = 0.01
N_PAYERS = 500
N_DAYS = 30
START_DATE = "2024-01-01"

In [5]:
sim = Cardsim(
    seed=SEED,
    fraud_rate=FRAUD_RATE,
    fraud_flag_threshold=FRAUD_FLAG_THRESHOLD,
    dcpc_folder=DCPC_DIR,
)

df = sim.simulate(
    n_payers=N_PAYERS,
    n_days=N_DAYS,
    start_date=START_DATE,
)

print(df.shape)
df.head()

Sourcing DCPC data
Sourcing successful; formatting data
DCPC data sourcing and formatting complete
Starting phase one: generating simulator world


Generated payment distributions

Generating payer profiles for 500 payers
Generated payer profiles

Generating payee profiles for 50 payees
Generated payee profiles

Calculated distance matrix

Generated world in 0 seconds

Starting phase two: generating transactions within world

Generated baseline transactions

Generated card type attribute

Generated location type attribute

Generated transaction amount

Generated and added payee distance
Generated transaction time

Generated fraud flag

Generated transactions in 0 seconds

Simulator completed in 0 seconds

(14583, 13)


,run_id,day_index,date,time_seconds,hour,date_time,payer_id,payee_id,credit_card,remote,distance,amount,fraud
0,S42P500D30,0,2024-01-01,62077,17,2024-01-01 17:14:37,0,29,0,1,60.415230,12.660533,0
1,S42P500D30,0,2024-01-01,72843,20,2024-01-01 20:14:03,2,18,0,1,82.807007,48.286849,0
2,S42P500D30,0,2024-01-01,25697,7,2024-01-01 07:08:17,2,16,0,1,110.113579,6.235227,0
3,S42P500D30,0,2024-01-01,71274,19,2024-01-01 19:47:54,3,47,0,0,123.713379,9.854808,0
4,S42P500D30,0,2024-01-01,25446,7,2024-01-01 07:04:06,3,45,0,1,148.600128,22.801516,0


## Table 1 — Probe results

Each quantity is classified by **origin**: *measured* (produced by the
simulator), *decided* (input parameter), or *derived* (computed by formula).

The fraud rate is **decided**, not measured: `fraud_flag_threshold = 0.01`
labels the top 1 % of transactions by risk score, so the observed 1.0012 % is
the empirical quantile over 14,583 discrete rows — a rounding artifact of a
discrete threshold, not a free estimate. The technical document lists it as
"measured" in Table 1 and as "decided" in Table 2; this notebook resolves the
inconsistency in favour of "decided".

In [6]:
table1 = pd.DataFrame(
    [
        ("Cards simulated",    f"{N_PAYERS}",                 "Decided (n_payers)"),
        ("Days simulated",     f"{N_DAYS}",                   f"Decided (n_days); {N_PAYERS*N_DAYS:,} card-day pairs"),
        ("Unique merchants",   f"{df['payee_id'].nunique()}", f"Derived: {N_PAYERS}/{N_PAYERS/df['payee_id'].nunique():g}"),
        ("Total transactions", f"{len(df):,}",                "Measured"),
        ("Fraud rate",         f"{df['fraud'].mean():.4%}",   "Decided (fraud_flag_threshold = 0.01)"),
        ("Raw columns",        f"{df.shape[1]}",              "CardSim internal schema"),
    ],
    columns=["Quantity", "Value", "Origin"],
).set_index("Quantity")
table1

,Value,Origin
Quantity,,
Cards simulated,500,Decided (n_payers)
Days simulated,30,"Decided (n_days); 15,000 card-day pairs"
Unique merchants,50,Derived: 500/10
Total transactions,"14,583",Measured
Fraud rate,1.0012%,Decided (fraud_flag_threshold = 0.01)
Raw columns,13,CardSim internal schema


**Conclusion.** The simulator loads the local DCPC CSVs and produces the
expected order of magnitude: ≈14.6k transactions from 15k card-day pairs, ~1 %
fraud labels, 50 merchants from the default 10:1 ratio. `scripts/probe_cardsim.py`
hard-codes the same configuration and prints the same quantities, so this
notebook and the script agree cell-for-line.

Because the probe resolves `DCPC_DIR` from `cardsim.__file__` rather than
downloading, it also proves offline reproducibility: the Atlanta Fed endpoint
is never contacted.

## Equation 3 — Empirical frequency per card-day

$$\hat{f} = \frac{\text{transactions}}{n_{\text{cards}} \times n_{\text{days}}}$$

CardSim does **not** expose this frequency as an input parameter. It emerges
from the DCPC calibration, which is precisely why it must be measured before
the base dataset can be sized.